# Widget Personalization Tool - Google Colab

Инструмент для массового удаления/добавления виджетов в JSON-конфигах.

## Возможности
- ✅ Массовое удаление виджетов по ID
- ✅ Добавление новых виджетов после указанного ID
- ✅ Автоматическое сравнение структуры файлов
- ✅ Валидация входных данных
- ✅ Отчёт о сегментах с ошибками

---


## 1. Установка зависимостей

Запустите эту ячейку для установки всех необходимых пакетов.


In [ ]:
# Установка зависимостей
!pip install pytest

print("✅ Зависимости установлены!")


## 2. Загрузка файлов проекта

Загрузите все файлы проекта или используйте встроенные файлы для демонстрации.


In [ ]:
import os
import json
from google.colab import files

# Создаём структуру папок
os.makedirs('input_jsons', exist_ok=True)
os.makedirs('output_jsons', exist_ok=True)
os.makedirs('processing', exist_ok=True)
os.makedirs('utils', exist_ok=True)
os.makedirs('widgets', exist_ok=True)
os.makedirs('tests', exist_ok=True)

print("📁 Структура папок создана!")


In [ ]:
# Создаём все файлы проекта

# main.py
main_py = '''from processing.pipeline import run_processing

if __name__ == "__main__":
    input_folder = "input_jsons"
    output_folder = "output_jsons"
    run_processing(input_folder, output_folder)
'''
with open('main.py', 'w', encoding='utf-8') as f:
    f.write(main_py)

# __init__.py files
for module in ['processing', 'utils', 'widgets']:
    with open(f'{module}/__init__.py', 'w') as f:
        f.write('')

print("📄 Основные файлы созданы!")


In [ ]:
# Создаём все модули проекта

# utils/io_utils.py
io_utils_code = '''import os
import json
from typing import List, Tuple, Dict, Any


def read_input_jsons(input_folder: str) -> Tuple[List[Tuple[str, Dict[str, Any]]], List[str]]:
    files_data: List[Tuple[str, Dict[str, Any]]] = []
    invalid_files: List[str] = []
    if not os.path.exists(input_folder):
        return files_data, invalid_files

    for filename in os.listdir(input_folder):
        if not filename.endswith(".json"):
            continue
        path = os.path.join(input_folder, filename)
        try:
            if os.path.getsize(path) == 0:
                print(f"Файл пустой: {filename}")
                invalid_files.append(filename)
                continue
        except OSError:
            continue

        try:
            with open(path, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except json.JSONDecodeError:
            print(f"Файл не в формате JSON или поврежден: {filename}")
            invalid_files.append(filename)
            continue
        files_data.append((filename, data))
    return files_data, invalid_files


def ensure_output_dir(output_folder: str) -> None:
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)


def write_single_output(output_folder: str, filename: str, data: Dict[str, Any]) -> None:
    new_filename = os.path.splitext(filename)[0] + "_new.json"
    output_path = os.path.join(output_folder, new_filename)
    with open(output_path, 'w', encoding='utf-8') as f_out:
        json.dump(data, f_out, ensure_ascii=False, indent=2)
    print(f"Обработан единственный файл. Создан новый файл: {new_filename}")


def write_merged_output(output_folder: str, merged_data: Dict[str, Any]) -> None:
    output_path = os.path.join(output_folder, "new_merged.json")
    with open(output_path, 'w', encoding='utf-8') as f_out:
        json.dump(merged_data, f_out, ensure_ascii=False, indent=2)
    print(f"Объединенный файл сохранен: {output_path}")


def write_multiple_outputs(output_folder: str, files_data: List[Tuple[str, Dict[str, Any]]]) -> None:
    print("Обнаружены различия в файлах. Создаются отдельные новые файлы:")
    for filename, data in files_data:
        new_filename = os.path.splitext(filename)[0] + "_new.json"
        output_path = os.path.join(output_folder, new_filename)
        with open(output_path, 'w', encoding='utf-8') as f_out:
            json.dump(data, f_out, ensure_ascii=False, indent=2)
        print(f"Создан новый файл: {new_filename}")
'''

with open('utils/io_utils.py', 'w', encoding='utf-8') as f:
    f.write(io_utils_code)

print("✅ utils/io_utils.py создан!")


In [ ]:
# utils/prompts.py
prompts_code = '''from typing import List, Dict, Any


def ask_action() -> str:
    print("Выберите действие:")
    print("1 - Удалить виджеты по ID")
    print("2 - Добавить виджеты после указанного ID")
    choice = input("Введите номер операции (1/2) и нажмите Enter: ").strip()
    while choice not in ("1", "2"):
        choice = input("Неверный ввод. Введите 1 или 2: ").strip()
    return "remove" if choice == "1" else "add"


def ask_ids_to_remove() -> List[str]:
    raw = input("Введите ID виджетов для удаления через запятую: ").strip()
    ids = [x.strip() for x in raw.split(",") if x.strip()]
    if not ids:
        print("ID не указаны. Действие будет пропущено.")
    return ids


def ask_widgets_to_add() -> Dict[str, Any]:
    after_id = input("После какого ID вставлять? Введите ID: ").strip()
    print("Введите JSON массив виджетов для добавления (пример: [{\\"id\\":\\"growthHackMain\\",\\"customizable\\":true,\\"title\\":\\"Виджет\\"}]):")
    widgets_json = input().strip()
    import json
    try:
        widgets = json.loads(widgets_json)
        if not isinstance(widgets, list):
            raise ValueError
    except Exception:
        print("Неверный JSON. Добавление будет пропущено.")
        widgets = []
    return {"after_id": after_id, "widgets": widgets}
'''

with open('utils/prompts.py', 'w', encoding='utf-8') as f:
    f.write(prompts_code)

print("✅ utils/prompts.py создан!")


In [ ]:
# widgets/operations.py
operations_code = '''import copy
from typing import Dict, Any, List, Tuple, Set, Iterable


def insert_widget_after(data: Dict[str, Any], after_id: str, widget_to_insert: Dict[str, Any]) -> Dict[str, Any]:
    for segment in data.get("widgets", []):
        widgets = segment.get("widgets", [])
        new_widgets = []
        for widget in widgets:
            new_widgets.append(widget)
            if widget.get("id") == after_id:
                new_widgets.append(copy.deepcopy(widget_to_insert))
        segment["widgets"] = new_widgets
    return data


def insert_widgets(data: Dict[str, Any], spec: Dict[str, Any]) -> Tuple[Dict[str, Any], List[str]]:
    after_id = (spec or {}).get("after_id", "")
    result = copy.deepcopy(data)
    missing_in_segments: List[str] = []

    # Even if there are no widgets to add (e.g., invalid JSON in prompt),
    # we still want to validate and report segments missing the anchor.
    if not spec or not spec.get("widgets"):
        if after_id:
            for segment in result.get("widgets", []):
                segment_widgets = segment.get("widgets", [])
                has_anchor = any(w.get("id") == after_id for w in segment_widgets)
                if not has_anchor:
                    missing_in_segments.append(segment.get("segment"))
        return result, missing_in_segments

    widgets_to_add: Iterable[Dict[str, Any]] = spec["widgets"]
    for segment in result.get("widgets", []):
        segment_widgets = segment.get("widgets", [])
        has_anchor = any(w.get("id") == after_id for w in segment_widgets)
        if not has_anchor:
            missing_in_segments.append(segment.get("segment"))
            continue
        # Insert each requested widget after the anchor, preserving order and duplicating after each anchor occurrence
        new_widgets: List[Dict[str, Any]] = []
        for w in segment_widgets:
            new_widgets.append(w)
            if w.get("id") == after_id:
                for to_add in widgets_to_add:
                    new_widgets.append(copy.deepcopy(to_add))
        segment["widgets"] = new_widgets

    return result, missing_in_segments


def remove_widgets(data: Dict[str, Any], widget_ids_to_remove: Set[str]) -> Dict[str, Any]:
    for segment in data.get("widgets", []):
        widgets = segment.get("widgets", [])
        segment["widgets"] = [w for w in widgets if w.get("id") not in widget_ids_to_remove]
    return data


def get_segments_full_signature(data: Dict[str, Any]) -> List[Tuple[str, Tuple[Tuple[str, Any, Any], ...]]]:
    signature: List[Tuple[str, Tuple[Tuple[str, Any, Any], ...]]] = []
    for seg in data.get("widgets", []):
        widgets_info = tuple((w.get("id"), w.get("customizable"), w.get("title")) for w in seg.get("widgets", []))
        signature.append((seg.get("segment"), widgets_info))
    return signature
'''

with open('widgets/operations.py', 'w', encoding='utf-8') as f:
    f.write(operations_code)

print("✅ widgets/operations.py создан!")


In [ ]:
# processing/pipeline.py
pipeline_code = '''import copy
from typing import List, Tuple, Dict, Any

from utils.io_utils import read_input_jsons, ensure_output_dir, write_single_output, write_merged_output, write_multiple_outputs
from utils.prompts import ask_action, ask_ids_to_remove, ask_widgets_to_add
from widgets.operations import remove_widgets, insert_widgets, get_segments_full_signature


def run_processing(input_folder: str, output_folder: str) -> None:
    files_data_raw = read_input_jsons(input_folder)
    # Backward compatibility if function signature hasn't been reloaded yet
    if isinstance(files_data_raw, tuple):
        files_data, invalid_files = files_data_raw
    else:
        files_data = files_data_raw  # type: ignore
        invalid_files = []

    if invalid_files:
        print("Найдены некорректные JSON файлы. Процесс остановлен. Исправьте следующие файлы:")
        for nf in invalid_files:
            print(f"- {nf}")
        return
    if len(files_data) == 0:
        print("Входные файлы не найдены или все файлы пустые/битые. Добавьте файлы в формате .json в папку input_jsons.")
        return

    action = ask_action()

    # Collect additional details based on action
    widget_ids_to_remove = []
    widgets_to_add_spec = None
    if action == "remove":
        widget_ids_to_remove = ask_ids_to_remove()
    elif action == "add":
        widgets_to_add_spec = ask_widgets_to_add()

    # Apply action per file
    processed: List[Tuple[str, Dict[str, Any]]] = []
    files_signatures: List[Any] = []
    for filename, data in files_data:
        if action == "remove":
            updated = remove_widgets(copy.deepcopy(data), set(widget_ids_to_remove))
        elif action == "add":
            updated, missing_segments = insert_widgets(copy.deepcopy(data), widgets_to_add_spec)
            if missing_segments:
                for seg in missing_segments:
                    print(f"{filename} | {seg} - не получилось добавить, т.к. отсутствует виджет \\"{widgets_to_add_spec.get('after_id','')}\\"")
        else:
            # no-op (should not happen due to prompt constraints)
            updated = copy.deepcopy(data)

        processed.append((filename, updated))
        files_signatures.append(get_segments_full_signature(updated))

    ensure_output_dir(output_folder)

    # Single file case
    if len(processed) == 1:
        write_single_output(output_folder, processed[0][0], processed[0][1])
        return

    # Compare signatures
    base_signature = files_signatures[0]
    all_match = all(sig == base_signature for sig in files_signatures[1:])

    if all_match:
        merged_data = copy.deepcopy(processed[0][1])
        write_merged_output(output_folder, merged_data)
    else:
        write_multiple_outputs(output_folder, processed)
'''

with open('processing/pipeline.py', 'w', encoding='utf-8') as f:
    f.write(pipeline_code)

print("✅ processing/pipeline.py создан!")


## 3. Демонстрационные файлы

Создаём примеры входных JSON файлов для демонстрации.


In [ ]:
# Создаём демонстрационные JSON файлы

demo_config1 = {
    "widgets": [
        {
            "segment": "header",
            "widgets": [
                {"id": "logo", "customizable": True, "title": "Логотип"},
                {"id": "menu", "customizable": True, "title": "Меню"},
                {"id": "search", "customizable": False, "title": "Поиск"}
            ]
        },
        {
            "segment": "content",
            "widgets": [
                {"id": "mainContent", "customizable": True, "title": "Основной контент"},
                {"id": "sidebar", "customizable": False, "title": "Боковая панель"}
            ]
        }
    ]
}

demo_config2 = {
    "widgets": [
        {
            "segment": "header",
            "widgets": [
                {"id": "logo", "customizable": True, "title": "Логотип"},
                {"id": "menu", "customizable": True, "title": "Меню"},
                {"id": "search", "customizable": False, "title": "Поиск"}
            ]
        },
        {
            "segment": "content",
            "widgets": [
                {"id": "mainContent", "customizable": True, "title": "Основной контент"},
                {"id": "sidebar", "customizable": False, "title": "Боковая панель"}
            ]
        }
    ]
}

# Сохраняем демо файлы
with open('input_jsons/config1.json', 'w', encoding='utf-8') as f:
    json.dump(demo_config1, f, ensure_ascii=False, indent=2)

with open('input_jsons/config2.json', 'w', encoding='utf-8') as f:
    json.dump(demo_config2, f, ensure_ascii=False, indent=2)

print("📄 Демонстрационные файлы созданы:")
print("  - input_jsons/config1.json")
print("  - input_jsons/config2.json")

# Показываем содержимое одного файла
print("\n📋 Пример содержимого config1.json:")
print(json.dumps(demo_config1, ensure_ascii=False, indent=2))


## 4. Загрузка собственных файлов (опционально)

Если у вас есть свои JSON файлы, загрузите их здесь. Иначе пропустите этот шаг.


In [ ]:
# Загрузка пользовательских файлов
print("📁 Загрузите ваши JSON файлы (опционально):")
print("Нажмите 'Browse' и выберите JSON файлы для обработки")

uploaded = files.upload()

# Перемещаем загруженные JSON файлы в input_jsons
for filename, content in uploaded.items():
    if filename.endswith('.json'):
        with open(f'input_jsons/{filename}', 'wb') as f:
            f.write(content)
        print(f"✅ {filename} загружен в input_jsons/")
    else:
        print(f"⚠️ {filename} пропущен (не JSON файл)")

# Показываем список всех файлов в input_jsons
if os.listdir('input_jsons'):
    print("\n📋 Файлы готовые к обработке:")
    for file in os.listdir('input_jsons'):
        print(f"  - {file}")
else:
    print("\n📋 Используются демонстрационные файлы")


## 5. Запуск инструмента

Теперь запустите основной инструмент. Следуйте интерактивным подсказкам.


In [ ]:
# Запуск основного инструмента
print("🚀 Запуск Widget Personalization Tool...")
print("=" * 50)

# Импортируем и запускаем
import sys
sys.path.append('.')

from processing.pipeline import run_processing

run_processing("input_jsons", "output_jsons")

print("\n" + "=" * 50)
print("✅ Обработка завершена!")


## 6. Просмотр результатов

Просмотрите созданные выходные файлы.


In [ ]:
# Показываем созданные файлы
if os.path.exists('output_jsons') and os.listdir('output_jsons'):
    print("📄 Созданные выходные файлы:")
    for file in os.listdir('output_jsons'):
        file_path = f'output_jsons/{file}'
        file_size = os.path.getsize(file_path)
        print(f"  - {file} ({file_size} bytes)")
        
        # Показываем содержимое первого файла
        if file.endswith('.json'):
            print(f"\n📋 Содержимое {file}:")
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = json.load(f)
                print(json.dumps(content, ensure_ascii=False, indent=2))
            except Exception as e:
                print(f"Ошибка чтения файла: {e}")
            break  # Показываем только первый файл
else:
    print("📭 Выходные файлы не найдены")


## 7. Скачивание результатов

Скачайте обработанные файлы на ваш компьютер.


In [ ]:
# Скачивание результатов
if os.path.exists('output_jsons') and os.listdir('output_jsons'):
    print("📥 Скачивание обработанных файлов:")
    
    for file in os.listdir('output_jsons'):
        file_path = f'output_jsons/{file}'
        files.download(file_path)
        print(f"✅ {file} готов к скачиванию")
else:
    print("📭 Нет файлов для скачивания")


## 📊 Итоги

### Что было сделано:
1. ✅ Установлены все зависимости
2. ✅ Создана полная структура проекта
3. ✅ Загружены демонстрационные или пользовательские файлы
4. ✅ Выполнена обработка виджетов
5. ✅ Просмотрены результаты
6. ✅ Скачаны обработанные файлы

### Возможности инструмента:
- 🗑️ **Удаление виджетов** по ID из нескольких файлов
- ➕ **Добавление виджетов** после указанного ID
- 🔍 **Автоматическое сравнение** структуры файлов
- ⚠️ **Валидация** входных данных с детальными сообщениями
- 📊 **Умные стратегии вывода**: один файл, merge или отдельные файлы

### Следующие шаги:
- Используйте скачанные файлы в вашем проекте
- При необходимости повторите процесс с новыми файлами
- Изучите код в созданных файлах для понимания логики

---

**🎉 Widget Personalization Tool успешно выполнен в Google Colab!**
